In [1]:
import sys
import os
current_path = os.path.abspath('')
data_path = '/'.join(current_path.split('/')[:-1]) + '/data/'
script_path = '/'.join(current_path.split('/')[:-1]) + '/scripts/'
sys.path.append(script_path)
sys.path.append('/home/fkunneman/.local/lib/python3.10/site-packages/')
sys.path.append('/usr/lib/python3/dist-packages/')

In [2]:
import torch
import gc
import importlib
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_community.vectorstores import Chroma
import chromadb
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings

import instruction_agent

/home/fkunneman/.local/lib/python3.10/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [3]:
### Load models for chatbot and rag in memory
with open('/home/fkunneman/hf.txt') as fin:
    hf = fin.read().strip()
os.environ["HF_TOKEN"] = hf
torch.cuda.empty_cache()
gc.collect()
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('BramVanroy/GEITje-7B-ultra', use_fast=True)
tokenizer.pad_token = tokenizer.eos_token # Ensure padding token is set
# Initialize language model
chatmodel = AutoModelForCausalLM.from_pretrained('BramVanroy/GEITje-7B-ultra', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")
#chatmodel = AutoModelForCausalLM.from_pretrained('robinsmits/Qwen1.5-7B-Dutch-Chat', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")     
# set pipeline
pipe = pipeline(
    "text-generation",
    model=chatmodel,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=250, # Limit the number of generated tokens for concise responses
    do_sample=True, # Enable sampling for more varied responses
    temperature=0.3 # Control creativity
)
llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [4]:
chroma_client = chromadb.Client()
#embedding_model = SentenceTransformer('jegormeister/bert-base-dutch-cased')
collection = chroma_client.create_collection(
    name="instruction_db2",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_construction": 30
        }
    }
)

In [5]:
instructions_travel = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_ov_stripped.csv'
instructions_passport = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_paspoort_stripped_v2.csv'
pat = data_path + 'opslag_inclusieve_spraakassistent_project/patterns_v2.csv'
qa_travel = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_ov_v5.csv'
qa_passport = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_paspoort.csv'
nav = data_path + 'opslag_inclusieve_spraakassistent_project/navigation.csv'


In [8]:
importlib.reload(instruction_agent)
assistant = instruction_agent.InstructAgent(llm,'travel')
assistant.prepare_patterns(pat)
assistant.prepare_instructions(instructions_travel)
assistant.setup_rag(collection,[['qa',qa_travel],['nav',nav]])

In [6]:
importlib.reload(instruction_agent)
assistant = instruction_agent.InstructAgent(llm,'passport')
assistant.prepare_instructions(instructions_passport)
assistant.prepare_patterns(pat)
assistant.setup_rag(collection,[['qa',qa_passport],['nav',nav]])

In [ ]:
assistant.chat()

Hallo! Ik ga je helpen met het plannen van een reis. Stel gerust een vraag als je er niet uit komt. Om te beginnen, zeg 'ik wil starten'.


You:  Start


{'ids': [['20dea474-3abc-4070-a000-539092644c6a', '81874ffc-bdb1-4a6f-a5ec-0d8278f805d9', 'ee164697-350c-4efa-841b-c431b1c75bc5']], 'embeddings': None, 'documents': [['Start', 'Quit', 'Quit']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'step_context': 'all', 'action': 'start', 'type': 'nav'}, {'step_context': 'all', 'type': 'nav', 'action': 'Done'}, {'type': 'nav', 'step_context': 'all', 'action': 'Done'}]], 'distances': [[0.0, 0.5145544409751892, 0.5145544409751892]]} b 80
Agent: Super! De eerste stap: onder de woorden 'Waar wil je heen?' staat een tekstvakje. Type daar het station in waar vandaan je wil vertrekken.


In [ ]:
klaar
